<a href="https://colab.research.google.com/github/M-Abbi/Probability-Statistics-Bootcamp/blob/main/Law_of_Large_Numbers_(LLN)_and_Central_Limit_Theorem_(CLT).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing the Law of Large Numbers (LLN) and Central Limit Theorem (CLT)

## 1. The Front Desk Scenario: High-Frequency Prop Trading Risk

Imagine you are running the Risk Management desk for a proprietary trading firm. One of your automated high-frequency trading (HFT) strategies has a highly asymmetric, non-normal payout distribution:
* It has an $80\%$ chance of a small, steady profit of **$10**

* It has a $20\%$ chance of a sharp, fat-tail loss of **-$35**

The true expected value ($\mathbb{E}[X]$) of any single trade is mathematically positive:
$$\mathbb{E}[X] = (10 \times 0.80) + (-35 \times 0.20) = 8 - 7 = \$1.00$$

The fund managers want to deploy this strategy across $500$ independent trading desks (or independent simulation universes). They need to visualize two crucial asymptotic behaviors to configure their risk parameters:
1. **Law of Large Numbers (LLN):** Prove that as a single desk executes more trades ($N \to 1000$), its running average profit per trade collapses directly onto the true mathematical expectation of $\$1.00$, stripping away localized noise.
2. **Central Limit Theorem (CLT):** Show that while the strategy's individual trade profile is wildly skewed and non-Gaussian, the distribution of *daily average performance* across all $500$ desks morphs into a perfect normal distribution centered around $\mathbb{E}[X]$ with a standard error that shrinks at a rate of $\frac{\sigma}{\sqrt{N}}$.

---

## 2. The Interactive Dashboard

Drag the interactive slider at the bottom of the dashboard to change the number of observations $N$.

* **On the Left (LLN):** Observe how the sample mean ($\bar{X}_N$) stabilizes directly on top of the red true expectation line as $N$ grows.
* **On the Right (CLT):** Watch the underlying binary distribution completely transform. The solid blue line tracks the changing **Sample Mean**, while the dashed crimson curve outlines the **Theoretical Normal Distribution** predicted by the CLT:
$$\bar{X}_N \sim N\left(\mu, \frac{\sigma^2}{N}\right)$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Setup the High-Frequency Trading Strategy Data (Run once)
payouts = np.array([10, -35])
probabilities = np.array([0.80, 0.20])
expected_value = np.sum(payouts * probabilities)  # True Expected Value ($1.00)

np.random.seed(42)
n_trades_max = 1000
n_days = 500

# Matrix of trade outcomes: rows = days (desks), columns = trade sequence
raw_trades = np.random.choice(payouts, size=(n_days, n_trades_max), p=probabilities)

# Pre-calculate the running cumulative average for Day 1 (for the LLN plot)
single_day_trades = raw_trades[0]
cumulative_averages = np.cumsum(single_day_trades) / np.arange(1, n_trades_max + 1)

# 2. Define the Custom Dashboard Output Function
def render_dashboard(N):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))
    fig.suptitle(f"Prop Trading Desk Simulation Dashboard (At N = {N} Trades)", fontsize=16, fontweight='bold')

    # --- LEFT GRAPH: Law of Large Numbers (LLN) ---
    ax1.plot(np.arange(1, N + 1), cumulative_averages[:N], color='darkblue', lw=2, label='Running Sample Average')
    ax1.axhline(expected_value, color='red', linestyle='--', linewidth=2, label=f'True Expected Value (${expected_value:.2f})')

    ax1.set_xlim(0, n_trades_max)
    ax1.set_ylim(-20, 15)
    ax1.set_title("Law of Large Numbers (LLN)\nRunning Average PnL of a Single Strategy")
    ax1.set_xlabel("Number of Trades Executed ($N$)")
    ax1.set_ylabel("Average PnL per Trade ($)")
    ax1.legend(loc="upper right")
    ax1.grid(True, alpha=0.3)

    # --- RIGHT GRAPH: Central Limit Theorem (CLT) ---
    # Calculate the average PnL of all 500 independent days up to trade N
    daily_averages_at_n = np.mean(raw_trades[:, :N], axis=1)
    current_sample_mean = np.mean(daily_averages_at_n)

    # Plot the empirical density histogram
    ax2.hist(daily_averages_at_n, bins=25, density=True, color='teal', alpha=0.5, edgecolor='white', label='Daily Averages')

    # Highlight the current sample mean with a solid blue line
    ax2.axvline(current_sample_mean, color='dodgerblue', linestyle='-', linewidth=2.5,
                label=f'Sample Mean (${current_sample_mean:.2f})')

    # If N is large enough, overlay the theoretical normal distribution curve
    if N > 1:
        population_std = np.std(raw_trades) # standard deviation of the underlying binary rule
        theoretical_std = population_std / np.sqrt(N)

        x_axis = np.linspace(-20, 15, 300)
        normal_curve = norm.pdf(x_axis, loc=expected_value, scale=theoretical_std)
        ax2.plot(x_axis, normal_curve, color='crimson', lw=2, linestyle='--', label='Theoretical CLT Fit')

    ax2.set_xlim(-20, 15)
    ax2.set_ylim(0, 0.5)
    ax2.set_title("Central Limit Theorem (CLT)\nDistribution of Daily Averages Across 500 Desks")
    ax2.set_xlabel("Average Daily PnL per Trade ($)")
    ax2.set_ylabel("Probability Density")
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 3. Create and Arrange Widgets Manually for Slider Placement
output_area = widgets.Output()

# Setup the slider widget layout
slider = widgets.IntSlider(
    value=1,
    min=1,
    max=1000,
    step=5,
    description='Trades ($N$):',
    continuous_update=True,
    layout=widgets.Layout(width='70%', margin='10px 0px 0px 50px')
)

# Callback function to clear the previous plot and render the updated snapshot
def on_value_change(change):
    with output_area:
        clear_output(wait=True)
        render_dashboard(change['new'])

# Link the slider to the callback function
slider.observe(on_value_change, names='value')

# Trigger the initial render at N=1
with output_area:
    render_dashboard(slider.value)

# Construct the layout box: Figures on TOP, Slider on BOTTOM
dashboard_layout = widgets.VBox([output_area, slider])
display(dashboard_layout)